<a href="https://colab.research.google.com/github/alinbui/claude_courses/blob/main/006_combined.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Building with Claude API — Accessing Claude Part 2 Notebook

This notebook combines four topics from *Building with Claude API* in order:

1. **System Prompts**
2. **Temperature**
3. **Response Streaming**
4. **Structured Data**

Each section pairs the relevant course text (as markdown) with runnable code cells.

## Setup

Shared dependencies, environment, client, and helpers used by every section below.

In [ ]:
# Install dependencies
%pip install anthropic

In [ ]:
# YOU WILL RUN THIS TO ENABLE THIS NOTEBOOK TO ACCESS ANTHROPIC API IN THE CLOUD
# Set up Google Colab to use my secret API key
import os
from google.colab import userdata
from anthropic import Anthropic

# This points to my secret API key
os.environ["ANTHROPIC_API_KEY"] = userdata.get('ANTHROPIC_API_KEY')

client = Anthropic()

In [ ]:
# Create an API client
from anthropic import Anthropic
client = Anthropic()
model = "claude-sonnet-4-6"

In [ ]:
# Multi-Turn Conversation: Helper functions

def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

## 1. System Prompts

> **System Prompts** = technique to customize Claude's response style and tone by assigning it a specific role or behavior pattern.
>
> **Purpose** — control *how* Claude responds rather than *what* it responds. Example: a math tutor role makes Claude give hints instead of direct answers.
>
> **Structure** — first line typically assigns a role ("You are a patient math tutor"), followed by specific behavioral instructions.
>
> **Key principle** — system prompts guide response *approach*, not content. The same question gets different treatment based on the assigned role.

**Technical implementation** — create a `params` dictionary, conditionally add the `system` key if a prompt is provided, then pass `params` to `create` with `**` unpacking. Handle the `None` case by excluding the `system` parameter entirely.

In [ ]:
def chat(messages, system=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

**Use case example** — a math tutor that gives guidance/hints rather than complete solutions. Compare Claude's response with and without the system prompt.

In [ ]:
messages = []

# Add the initial user question
add_user_message(messages, "What's 12 to the power of 3?")

# Without system prompt
answer = chat(messages)
print(answer)
print("NEXT EXECUTION")

# With system prompt
system = """
You are a patient math tutor.
Do not directly answer a student's questions.
Guide them to a solution step by step.
"""

add_user_message(messages, "What's 12 to the power of 3?")

answer = chat(messages, system=system)
print(answer)

## Calculating 12³

12³ = 12 × 12 × 12

- 12 × 12 = **144**
- 144 × 12 = **1,728**

**12³ = 1,728**
NEXT EXECUTION
Let's work through this together step by step!

First, do you know what it means to raise a number to a **power**?

When we say 12 to the power of 3, it means we **multiply 12 by itself 3 times**. So it looks like this:

> 12 × 12 × 12

Can you try to solve that? Start with the first part:

**What is 12 × 12?** 😊


**Try it yourself** — add a system prompt to the cell below to reduce the amount of code Claude generates (e.g. *"You are a terse senior engineer. Reply with only the function — no comments, no surrounding explanation."*).

In [ ]:
# Update this cell with a system prompt to reduce the amount of code generated

messages = []

add_user_message(
   messages,
   "Write a Python function that checks a string for duplicate characters."
)

answer = chat(messages)

print(answer)

## Checking for Duplicate Characters in a String

Here's a Python function that checks a string for duplicate characters, along with several variations for different use cases:

```python
def has_duplicates(string: str) -> bool:
    """
    Check if a string contains any duplicate characters.

    Args:
        string: The input string to check.

    Returns:
        True if duplicates exist, False otherwise.
    """
    return len(string) != len(set(string))


def find_duplicates(string: str) -> set:
    """
    Find and return all duplicate characters in a string.

    Args:
        string: The input string to check.

    Returns:
        A set of characters that appear more than once.
    """
    seen = set()
    duplicates = set()

    for char in string:
        if char in seen:
            duplicates.add(char)
        else:
            seen.add(char)

    return duplicates


def get_duplicate_details(string: str) -> dict:
    """
    Get detailed information about duplicate chara

## 2. Temperature

> **Temperature** = parameter (0–1) that controls randomness in Claude's text generation by influencing token selection probabilities.
>
> **Text generation process**: Input text → tokenization → probability assignment to possible next tokens → token selection based on probabilities → repeat.
>
> **Temperature effects**:
> - Temperature `0` = deterministic output, always selects the highest-probability token
> - Higher temperature = increases chance of selecting lower-probability tokens; more creative/unexpected outputs
>
> **Usage guidelines**:
> - Low temperature (near `0`) = data extraction, factual tasks requiring consistency
> - High temperature (near `1`) = creative tasks like brainstorming, writing, jokes, marketing
>
> **Key insight** — temperature directly manipulates the probability distribution of next-token selection, making high-probability tokens more or less dominant in the selection process.

**Implementation** — add a `temperature` parameter to `chat()`. Higher values don't *guarantee* different outputs; they just raise the probability of variation.

In [ ]:
def chat(messages, system=None, temperature=1.0):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

**Example** — a creative task (movie ideas) benefits from higher temperature. Re-run this cell several times at `temperature=1.0` and again at `temperature=0.0` to feel the difference.

In [ ]:
messages = []

add_user_message(
    messages,
    "Generate a one-sentence movie idea."
)

answer = chat(messages, temperature=1.0)

answer

"Here's a movie idea:\n\n**A retired safecracker with dementia must break into a high-security vault to recover a mysterious box he hid there decades ago — before his memory disappears completely and takes the combination with it.**"

## 3. Response Streaming

> **Response Streaming** = technique to display AI responses chunk-by-chunk as they're generated instead of waiting for the complete response.
>
> **Problem solved** — AI responses can take 10–30 seconds. Users expect immediate feedback, not just spinners.
>
> **How it works**:
> 1. Server sends user message to Claude
> 2. Claude immediately sends initial response (no text, just acknowledgment)
> 3. Stream of events follows, each containing text chunks
> 4. Server forwards chunks to the frontend for real-time display

**Event types**:
- `message_start` — initial acknowledgment
- `content_block_start` — text generation begins
- `content_block_delta` — contains actual text chunks (most important)
- `content_block_stop` / `message_stop` — generation complete

**Basic implementation** — pass `stream=True` to `client.messages.create()` and iterate over the returned event iterator. Useful when you want full visibility into the raw stream events.

In [ ]:
messages = []
add_user_message(messages, "Write a 1 sentence description of a fake database")

stream = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    stream=True
)

for event in stream:
    print(event)

RawMessageStartEvent(message=Message(id='msg_01RyCmsWHYXBwSNh2KbRoxRv', container=None, content=[], model='claude-sonnet-4-6', role='assistant', stop_details=None, stop_reason=None, stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='global', input_tokens=18, output_tokens=1, server_tool_use=None, service_tier='standard')), type='message_start')
RawContentBlockStartEvent(content_block=TextBlock(citations=None, text='', type='text'), index=0, type='content_block_start')
RawContentBlockDeltaEvent(delta=TextDelta(text='Here', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text=' is a one sentence description of a fake database:\n\n**"NutriTra', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text='ck Pro Database"** is a fictional nu

**Simplified** — `client.messages.stream()` is a context manager with a `text_stream` property that yields just the text chunks, hiding the event-type machinery. This is what you typically want for rendering tokens to a UI.

In [ ]:
messages = []
add_user_message(messages, "Write a 1 sentence description of a fake database")

with client.messages.stream(
    model=model,
    max_tokens=1000,
    messages=messages
) as stream:
    for text in stream.text_stream:
        print(text, end="")

Here is a one sentence description of a fake database:

**"NutriTrack Pro Database"** is a fictional nutritional database containing detailed macro and micronutrient information for over 500,000 food items, recipes, and restaurant menu options from around the world.

**Capturing the final message** — after streaming finishes, `stream.get_final_message()` assembles all chunks into the same `Message` object a non-streaming call would return. Use this to persist the response (database storage, history, etc.) while still streaming chunks to the client in real time.

**Key benefits** — better UX through immediate response visibility, plus complete message capture for storage.

In [ ]:
messages = []
add_user_message(messages, "Write a 1 sentence description of a fake database")

with client.messages.stream(
    model=model,
    max_tokens=1000,
    messages=messages
) as stream:
    for text in stream.text_stream:
        # Send each chunk to your client
        pass

    # Get the complete message for database storage
    final_message = stream.get_final_message()
    print(final_message)

ParsedMessage(id='msg_013XhUAv626f8SMwMxJGVY3Z', container=None, content=[ParsedTextBlock(citations=None, text='"CustomerVault is a fictional relational database containing fabricated records of 10,000 imaginary customers, including made-up names, addresses, purchase histories, and account details, used solely for demonstration and testing purposes."', type='text', parsed_output=None)], model='claude-sonnet-4-6', role='assistant', stop_details=None, stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='global', input_tokens=18, output_tokens=50, server_tool_use=None, service_tier='standard'))


## 4. Structured Data

> **Structured Data Generation** = technique using assistant message prefilling + stop sequences to get raw output without Claude's natural explanatory headers/footers.
>
> **Problem** — Claude automatically adds markdown formatting, headers, and commentary when generating JSON/code/structured content. Users often want just the raw data for copy/paste or programmatic use.
>
> **Solution pattern**:
> 1. **User message** = request for structured data
> 2. **Assistant message prefill** = opening delimiter (e.g. ` ```json `)
> 3. **Stop sequence** = closing delimiter (e.g. ` ``` `)
>
> **How it works** — Claude sees the prefilled message, assumes it has already started its response, generates only the requested content, and stops when hitting the delimiter.
>
> **Result** — raw structured data output with no extra formatting or commentary.
>
> **Application** — works for any structured data type (JSON, Python code, lists, etc.), not just JSON. Use whenever you need clean, parseable output without explanatory text.

> ⚠️ **Note:** prefill is deprecated in the newest models — see the model-by-model breakdown immediately below.

> ### Heads up — assistant message prefill is deprecated in newer models
>
> The "prefill the assistant turn with ` ```json ` and stop at ` ``` `" pattern described above **no longer works on the newest Claude models**. The API rejects it with:
>
> ```
> BadRequestError: 400 - This model does not support assistant message prefill.
> The conversation must end with a user message.
> ```
>
> **Models that still support prefill (older):**
> - `claude-3-5-sonnet-*`, `claude-3-7-sonnet-*`
> - `claude-sonnet-4-0`, `claude-sonnet-4-5`
> - `claude-opus-4-0`, `claude-opus-4-1`, `claude-opus-4-5`
> - `claude-haiku-3-*`
>
> **Models that reject prefill (newer):**
> - `claude-sonnet-4-6` *(confirmed empirically)*
> - `claude-opus-4-6`, `claude-opus-4-7` *(same generation — assume rejected)*
> - `claude-haiku-4-5` *(same generation — assume rejected)*
>
> **What I updated in the Structured Data exercises below:**
> 1. The JSON example **does not use `add_assistant_message(messages, "```json")`** anymore. Instead, the formatting constraint ("Respond with ONLY valid JSON…") is pushed into the **user** message. This works identically on every model.
> 2. The `stop_sequences` demo uses a **prime-numbers** example (`stop_sequences=["7"]`) instead of the original JSON-fence pattern, since the fence pattern only made sense when paired with prefill.
> 3. `chat()` was extended with a `stop_sequences` parameter but no longer relies on prefill to be useful.
>
> If you specifically want to learn the prefill pattern, change the `model = "..."` cell in **Setup** to one of the older models listed above (e.g. `claude-opus-4-5`) and uncomment any prefill-based examples you add.

**Baseline — no `stop_sequences`** — the model produces JSON but wraps it in prose and code fences.

In [ ]:
messages = []

add_user_message(messages, "Generate a very short event bridge as a json")

chat(messages)

'Here\'s a very short EventBridge event as JSON:\n\n```json\n{\n  "source": "my.app",\n  "detail-type": "UserCreated",\n  "detail": {\n    "userId": "123",\n    "email": "user@example.com"\n  }\n}\n```\n\nThis is a minimal custom EventBridge event with:\n- **source** – who sent it\n- **detail-type** – what happened\n- **detail** – the event data'

**Updated `chat` helper** — extends `chat()` with a `stop_sequences` parameter that forwards custom stop strings to the API.

In [ ]:
def chat(messages, system=None, stop_sequences=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
    }
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    message = client.messages.create(**params)
    return message.content[0].text

**Cleaner pattern (works on every model)** — push the formatting constraint into the user prompt, then `json.loads` the response directly. No prefill, no fence-stripping.

In [ ]:
import json

messages = []
add_user_message(
    messages,
    "Generate a very short event bridge as a json. "
    "Respond with ONLY valid JSON. No markdown fences, no prose, no explanation."
)

text = chat(messages)
print(text)

{"version":"0","id":"a1b2c3d4-e5f6-7890-abcd-ef1234567890","detail-type":"UserCreated","source":"com.myapp.users","account":"123456789012","time":"2024-01-15T10:30:00Z","region":"us-east-1","resources":[],"detail":{"userId":"u-001","email":"user@example.com","status":"active"}}


In [ ]:
data = json.loads(text)
data

{'version': '0',
 'id': 'a1b2c3d4-e5f6-7890-abcd-ef1234567890',
 'detail-type': 'UserCreated',
 'source': 'com.myapp.users',
 'account': '123456789012',
 'time': '2024-01-15T10:30:00Z',
 'region': 'us-east-1',
 'resources': [],
 'detail': {'userId': 'u-001',
  'email': 'user@example.com',
  'status': 'active'}}

### Stop sequences in action

**Stop sequences** = force Claude to halt generation when a specific string appears. The matched string is **not** included in the final output.

A clear demo: ask for the first 10 prime numbers, but pass `stop_sequences=["7"]`. The response cuts off the instant Claude tries to write `7`.

In [ ]:
messages = []
add_user_message(messages, "List the first 10 prime numbers, one per line.")

text = chat(messages, stop_sequences=["7"])
print(text)

Here are the first 10 prime numbers:

2
3
5



**Inspecting `stop_reason`** — `chat()` only returns text, which loses the metadata. Call `client.messages.create` directly when you want to see *why* generation ended. `stop_reason` will be `"stop_sequence"` when one of your stop strings fired, and `stop_sequence` tells you which one.

In [ ]:
messages = []
add_user_message(messages, "List the first 10 prime numbers, one per line.")

message = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    stop_sequences=["7"],
)

print(message.content[0].text)
print("---")
print("stop_reason:", message.stop_reason)
print("stop_sequence:", message.stop_sequence)

Here are the first 10 prime numbers:

2
3
5

---
stop_reason: stop_sequence
stop_sequence: 7
